## Portfolio Analysis Agents for Financial Data

This notebook demonstrates creating multiple Foundry agents for portfolio analysis, each managing a specific client's portfolio of stocks and mutual funds.

### Installing Required Libraries

In [ ]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity yfinance requests pandas numpy urllib3

### Importing Required Libraries

In [ ]:
import os
import json
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
import yfinance as yf
import pandas as pd
import requests
import urllib3
from datetime import datetime

# Disable warnings for unverified HTTPS
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

load_dotenv()

### Setting Up the Environment Variables and Foundry Client

In [ ]:
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

# Initialize Foundry Project Client
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

# Create OpenAI Client
openai_client = project_client.get_openai_client()

### Define Client Portfolio Data

In [ ]:
# Hardcoded client portfolio data
client_portfolios = {
    "Client_1": {
        "name": "Client 1",
        "stocks": ["TCS.NS", "WIPRO.NS", "INFY.NS", "HDFCBANK.NS"],
        "mutual_funds": ["119551", "119022"]  # HDFC Nifty 50 Index Fund, SBI Bluechip Fund (scheme codes)
    },
    "Client_2": {
        "name": "Client 2",
        "stocks": ["RELIANCE.NS", "HDFCBANK.NS", "ICICIBANK.NS", "LT.NS"],
        "mutual_funds": ["119551", "120485"]  # HDFC Nifty 50 Index Fund, ICICI Prudential Bluechip Fund
    },
    "Client_3": {
        "name": "Client 3",
        "stocks": ["TATAMOTORS.NS", "ADANIGREEN.NS", "ETERNAL.NS", "PAYTM.NS"],
        "mutual_funds": ["119598", "119618"]  # Quant Small Cap Fund, Mirae Asset Emerging Bluechip Fund
    },
    "Client_4": {
        "name": "Client 4",
        "stocks": ["ITC.NS", "COALINDIA.NS", "NTPC.NS", "POWERGRID.NS"],
        "mutual_funds": ["119547"]  # HDFC Dividend Yield Fund
    },
    "Client_5": {
        "name": "Client 5",
        "stocks": ["INFY.NS", "SUNPHARMA.NS", "BHARTIARTL.NS", "TATACONSUM.NS"],
        "mutual_funds": ["119510", "120303"]  # ICICI Prudential Technology Fund, Motilal Oswal Nasdaq 100 ETF
    }
}

print("Client portfolios loaded:")
for client_id, portfolio in client_portfolios.items():
    print(f"{portfolio['name']}: {len(portfolio['stocks'])} stocks, {len(portfolio['mutual_funds'])} mutual funds")

### Data Fetching Functions

In [ ]:
def get_nse_stock_data(symbol):
    """Get NSE stock data using Yahoo Finance with key metrics"""
    try:
        ticker = yf.Ticker(f"{symbol}")
        info = ticker.info
        
        # Extract key metrics
        current_price = info.get("currentPrice", "N/A")
        fifty_two_week_high = info.get("fiftyTwoWeekHigh", "N/A")
        fifty_two_week_low = info.get("fiftyTwoWeekLow", "N/A")
        market_cap = info.get("marketCap", "N/A")
        pe_ratio = info.get("trailingPE", "N/A")
        dividend_yield = info.get("dividendYield", "N/A")
        
        return {
            'success': True,
            'ticker': symbol,
            'current_price': current_price,
            'fifty_two_week_high': fifty_two_week_high,
            'fifty_two_week_low': fifty_two_week_low,
            'market_cap': market_cap,
            'pe_ratio': pe_ratio,
            'dividend_yield': dividend_yield,
            'company_name': info.get('longName', 'N/A')
        }
    except Exception as e:
        return {"success": False, "ticker": symbol, "error": str(e)}

def get_fund_data(scheme_code):
    """Fetch mutual fund data from MFAPI.in"""
    try:
        url = f"https://api.mfapi.in/mf/{scheme_code}"
        response = requests.get(url, verify=False, timeout=10)
        
        if response.status_code != 200:
            return {"success": False, "scheme_code": scheme_code, "error": "Could not fetch data"}
        
        data = response.json()
        meta = data.get("meta", {})
        nav_list = data.get("data", [])
        
        # Get latest NAV
        latest_nav = nav_list[0] if nav_list else {}
        
        return {
            'success': True,
            'scheme_code': scheme_code,
            'fund_name': meta.get('schemeName', 'N/A'),
            'fund_type': meta.get('schemeType', 'N/A'),
            'latest_nav': latest_nav.get('nav', 'N/A'),
            'nav_date': latest_nav.get('date', 'N/A'),
            'category': meta.get('schemeCategory', 'N/A')
        }
    except Exception as e:
        return {"success": False, "scheme_code": scheme_code, "error": str(e)}

def fetch_portfolio_summary(client_id):
    """Fetch summary of stocks and mutual funds for a client"""
    portfolio = client_portfolios.get(client_id)
    if not portfolio:
        return {"error": "Client not found"}
    
    # Fetch stock data
    stocks_data = []
    for stock in portfolio['stocks']:
        stock_info = get_nse_stock_data(stock)
        stocks_data.append(stock_info)
    
    # Fetch mutual fund data
    funds_data = []
    for fund_code in portfolio['mutual_funds']:
        fund_info = get_fund_data(fund_code)
        funds_data.append(fund_info)
    
    return {
        "client_id": client_id,
        "client_name": portfolio['name'],
        "stocks": stocks_data,
        "mutual_funds": funds_data
    }

print("Data fetching functions defined")

### Create Portfolio Analysis Tool Descriptions

In [ ]:
def get_portfolio_tool_definition(client_id):
    """Generate tool definition for portfolio analysis"""
    portfolio = client_portfolios.get(client_id)
    
    if not portfolio:
        return None
    
    stocks_list = ", ".join(portfolio['stocks'])
    
    tool_description = f"""
    You are a portfolio analysis assistant for {portfolio['name']}.
    
    Portfolio Holdings:
    - Stocks: {stocks_list}
    - Mutual Funds: {len(portfolio['mutual_funds'])} funds
    
    Your responsibilities:
    1. Provide real-time stock price information and technical metrics
    2. Analyze portfolio diversification and risk exposure
    3. Give insights on individual security performance
    4. Recommend portfolio rebalancing when appropriate
    5. Answer questions about market trends and economic indicators
    
    Use data from Yahoo Finance and MFAPI to provide accurate information.
    """
    
    return tool_description.strip()

# Test
print("Tool definition sample for Client 1:")
print(get_portfolio_tool_definition("Client_1"))

### Create Portfolio Analysis Agents

In [ ]:
# Create agents for each client
agents = {}

for client_id, portfolio in client_portfolios.items():
    agent_name = f"portfolio-agent-{client_id.lower()}"
    instructions = get_portfolio_tool_definition(client_id)
    
    agent = project_client.agents.create_version(
        agent_name=agent_name,
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions=instructions
        )
    )
    
    agents[client_id] = {
        "agent": agent,
        "agent_name": agent_name,
        "portfolio": portfolio
    }
    
    print(f"✅ Created agent: {agent_name}")
    print(f"   Agent ID: {agent.id}")
    print()

### Create Conversations for Each Client's Agent

In [ ]:
# Create conversations for each agent
conversations = {}

for client_id, agent_info in agents.items():
    conversation = openai_client.conversations.create()
    conversations[client_id] = conversation.id
    print(f"Created conversation for {agent_info['portfolio']['name']}: {conversation.id}")

### Query Portfolio Agents - Client 1

In [ ]:
# Example: Query Client 1's portfolio agent
client_id = "Client_1"
user_query = "What are the current stock prices in my portfolio and how are they performing?"

response = openai_client.responses.create(
    conversation=conversations[client_id],
    extra_body={
        "agent": {
            "name": agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Query: {user_query}")
print(f"\nAgent Response:\n{response.output_text}")

### Query Portfolio Agents - Client 2

In [ ]:
# Example: Query Client 2's portfolio agent
client_id = "Client_2"
user_query = "Analyze my portfolio diversification. What sectors am I exposed to?"

response = openai_client.responses.create(
    conversation=conversations[client_id],
    extra_body={
        "agent": {
            "name": agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Query: {user_query}")
print(f"\nAgent Response:\n{response.output_text}")

### Query Portfolio Agents - Client 3

In [ ]:
# Example: Query Client 3's portfolio agent
client_id = "Client_3"
user_query = "What are the latest performance metrics for my holdings and should I rebalance?"

response = openai_client.responses.create(
    conversation=conversations[client_id],
    extra_body={
        "agent": {
            "name": agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Query: {user_query}")
print(f"\nAgent Response:\n{response.output_text}")

### Query Portfolio Agents - Client 4

In [ ]:
# Example: Query Client 4's portfolio agent
client_id = "Client_4"
user_query = "Give me a summary of my mutual fund investments and their NAV trends."

response = openai_client.responses.create(
    conversation=conversations[client_id],
    extra_body={
        "agent": {
            "name": agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Query: {user_query}")
print(f"\nAgent Response:\n{response.output_text}")

### Query Portfolio Agents - Client 5

In [ ]:
# Example: Query Client 5's portfolio agent
client_id = "Client_5"
user_query = "What is the technology sector exposure in my portfolio and how is it trending?"

response = openai_client.responses.create(
    conversation=conversations[client_id],
    extra_body={
        "agent": {
            "name": agents[client_id]["agent_name"],
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Query: {user_query}")
print(f"\nAgent Response:\n{response.output_text}")

### Fetch Live Portfolio Data Example

In [ ]:
# Fetch complete portfolio summary for a client
print("Fetching portfolio summary for Client 1...\n")
portfolio_summary = fetch_portfolio_summary("Client_1")

print(f"Client: {portfolio_summary['client_name']}")
print(f"\nStocks:")
for stock in portfolio_summary['stocks']:
    if stock['success']:
        print(f"  {stock['ticker']}: ₹{stock['current_price']} ({stock['company_name']})")
    else:
        print(f"  {stock['ticker']}: Error - {stock['error']}")

print(f"\nMutual Funds:")
for fund in portfolio_summary['mutual_funds']:
    if fund['success']:
        print(f"  {fund['fund_name']}: NAV ₹{fund['latest_nav']} (as of {fund['nav_date']})")
    else:
        print(f"  Scheme {fund['scheme_code']}: Error - {fund['error']}")

### Agent Management Summary

In [ ]:
print("="*60)
print("PORTFOLIO AGENTS SUMMARY")
print("="*60)

for client_id, agent_info in agents.items():
    portfolio = agent_info['portfolio']
    print(f"\n{portfolio['name']}")
    print(f"  Agent Name: {agent_info['agent_name']}")
    print(f"  Agent ID: {agent_info['agent']['id']}")
    print(f"  Conversation ID: {conversations[client_id]}")
    print(f"  Holdings: {len(portfolio['stocks'])} stocks, {len(portfolio['mutual_funds'])} mutual funds")
    print(f"  Stocks: {', '.join(portfolio['stocks'])}")

print(f"\n{'='*60}")
print(f"Total Agents Created: {len(agents)}")
print(f"{'='*60}")